In [3]:
#!/usr/bin/env python3
from __future__ import annotations

import argparse
from pathlib import Path
import pandas as pd


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Analyze exp_022 clusterbase output CSVs")
    parser.add_argument(
        "--results-dir",
        default="results",
        help="Directory containing clusterbase_points.csv and clusterbase_batches.csv",
    )
    parser.add_argument(
        "--top-k",
        type=int,
        default=10,
        help="Number of best/worst datasets to print",
    )
    # In notebook environments (ipykernel), unknown args like "-f <kernel.json>"
    # are injected into argv. parse_known_args keeps CLI behavior while avoiding
    # crashes when users run this file/cell from Jupyter.
    args, _ = parser.parse_known_args()
    return args


def require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")


def main() -> None:
    args = parse_args()
    results_dir = Path(args.results_dir)

    points_path = results_dir / "clusterbase_points.csv"
    batches_path = results_dir / "clusterbase_batches.csv"
    summary_path = results_dir / "clusterbase_summary_by_dataset.csv"

    require_file(points_path)
    require_file(batches_path)

    points = pd.read_csv(points_path)
    batches = pd.read_csv(batches_path)
    summary = pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame()

    ok_batches = batches[batches.get("status", "ok") == "ok"].copy()
    if ok_batches.empty:
        raise ValueError("No successful batches found (status == 'ok').")

    global_metrics = pd.Series(
        {
            "n_datasets": ok_batches["dataset"].nunique(),
            "n_batches": len(ok_batches),
            "mean_abs_error": ok_batches["abs_error"].mean(),
            "median_abs_error": ok_batches["abs_error"].median(),
            "p90_abs_error": ok_batches["abs_error"].quantile(0.90),
            "mean_instance_accuracy": ok_batches["instance_accuracy"].mean(),
            "median_instance_accuracy": ok_batches["instance_accuracy"].median(),
        }
    )

    if "status" in batches.columns:
        ok_batches = batches[batches["status"] == "ok"].copy()
    else:
        ok_batches = batches.copy()
    if ok_batches.empty:
        raise ValueError("No successful batches found (status == 'ok').")
    
    if summary.empty:
        summary = (
            ok_batches.groupby("dataset", as_index=False)
            .agg(
                n_batches=("batch_id", "count"),
                mean_abs_error=("abs_error", "mean"),
                median_abs_error=("abs_error", "median"),
                mean_instance_acc=("instance_accuracy", "mean"),
            )
            .sort_values("mean_abs_error")
        )

    err_by_prev = (
        ok_batches.groupby("target_prev_pos", as_index=False)
        .agg(
            mean_abs_error=("abs_error", "mean"),
            median_abs_error=("abs_error", "median"),
            mean_acc=("instance_accuracy", "mean"),
            n=("batch_id", "count"),
        )
        .sort_values("target_prev_pos")
    )

    points = points.copy()
    points["is_correct"] = (points["true_label"] == points["pred_label"]).astype(int)
    points["decision_margin"] = points["dist_to_centroid0"] - points["dist_to_centroid1"]

    analysis_dir = results_dir / "analysis_exports"
    analysis_dir.mkdir(parents=True, exist_ok=True)

    worst_batches = ok_batches.sort_values("abs_error", ascending=False).head(200)
    worst_batches.to_csv(analysis_dir / "worst_batches_top200.csv", index=False)

    key_cols = ["dataset", "repeat", "batch_id"]
    bad_keys = worst_batches[key_cols].drop_duplicates()
    bad_points = points.merge(bad_keys, on=key_cols, how="inner")
    bad_points.to_csv(analysis_dir / "points_from_worst_batches.csv", index=False)

    global_metrics.to_frame("value").to_csv(analysis_dir / "global_metrics.csv")
    summary.sort_values("mean_abs_error").to_csv(analysis_dir / "dataset_ranking.csv", index=False)
    err_by_prev.to_csv(analysis_dir / "error_by_prevalence.csv", index=False)

    print("=== Global metrics ===")
    print(global_metrics.to_string())
    print("\n=== Top datasets (lowest mean_abs_error) ===")
    print(summary.sort_values("mean_abs_error").head(args.top_k).to_string(index=False))
    print("\n=== Worst datasets (highest mean_abs_error) ===")
    print(summary.sort_values("mean_abs_error", ascending=False).head(args.top_k).to_string(index=False))
    print(f"\nAnalysis exports written to: {analysis_dir}")


if __name__ == "__main__":
    main()

=== Global metrics ===
n_datasets                     30.000000
n_batches                   17100.000000
mean_abs_error                  0.141314
median_abs_error                0.110000
p90_abs_error                   0.320000
mean_instance_accuracy          0.745806
median_instance_accuracy        0.730000

=== Top datasets (lowest mean_abs_error) ===
      dataset  n_batches  mean_abs_error  median_abs_error  mean_instance_acc
       iris.1        570        0.000000             0.000           1.000000
       wine.2        570        0.000000             0.000           1.000000
breast-cancer        570        0.014193             0.010           0.985807
       wine.1        570        0.039754             0.040           0.929404
         wdbc        570        0.051807             0.050           0.901877
       wine.3        570        0.062842             0.060           0.937158
    balance.1        570        0.075930             0.070           0.859193
     spambase       